In [1]:
import site
site.addsitedir("/home/jovyan/vault/heidelberg_lulc_2025/pylibs/lib/python3.11/site-packages")

import os, re, glob
import numpy as np
import pandas as pd
import rasterio

PROJ = "/home/jovyan/vault/heidelberg_lulc_2025"
OUT = f"{PROJ}/outputs/inspection"
os.makedirs(OUT, exist_ok=True)

MONTHS = {m: i for i, m in enumerate(
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], start=1)}

def parse_name(fname):
    """Return (dataset, month_number, resolution_m) from the file name."""
    m = re.match(r"S1_2025_([A-Za-z]{3})_Orbit15_(10|40)m\.tif$", fname)
    if m:
        return "S1", MONTHS[m.group(1)], int(m.group(2))
    m = re.match(r"DW_2025_(\d{2})_(10|40)m_9class\.tif$", fname)
    if m:
        return "DW", int(m.group(1)), int(m.group(2))
    return None, None, None

files = sorted(glob.glob(f"{PROJ}/sentinel1/raw/*.tif") + glob.glob(f"{PROJ}/dynamic_world/raw/*.tif"))
file_rows, band_rows = [], []

for path in files:
    fname = os.path.basename(path)
    ds, month, res = parse_name(fname)
    if ds is None:
        print("SKIPPED (unexpected name):", fname)
        continue

    with rasterio.open(path) as src:
        t = src.transform
        b = src.bounds
        file_rows.append({
            "dataset": ds, "res_m": res, "month": month, "file": fname,
            "size_MB": round(os.path.getsize(path) / 1e6, 2),
            "bands": src.count,
            "descriptions": "|".join(d if d else "NONE" for d in src.descriptions),
            "dtypes": "|".join(sorted(set(src.dtypes))),
            "crs": src.crs.to_string() if src.crs else "NONE",
            "width": src.width, "height": src.height,
            "pixel_x": t.a, "pixel_y": t.e, "rot_b": t.b, "rot_d": t.d,
            "origin_x": t.c, "origin_y": t.f,
            "left": b.left, "bottom": b.bottom, "right": b.right, "top": b.top,
            "nodata": src.nodata,
            "compression": src.profile.get("compress"),
        })

        for i in range(1, src.count + 1):
            name = src.descriptions[i - 1] or f"band{i}"
            a = src.read(i)
            invalid = ~np.isfinite(a)
            if src.nodata is not None and np.isfinite(src.nodata):
                invalid |= (a == src.nodata)
            v = a[~invalid].astype(np.float64)

            row = {"dataset": ds, "res_m": res, "month": month, "file": fname,
                   "band": i, "name": name, "dtype": src.dtypes[i - 1],
                   "n_pixels": a.size, "n_invalid": int(invalid.sum()),
                   "pct_invalid": round(100 * invalid.mean(), 3)}
            if v.size:
                p1, p50, p99 = np.percentile(v, [1, 50, 99])
                row.update({"min": v.min(), "p1": p1, "median": p50, "p99": p99,
                            "max": v.max(), "mean": v.mean()})
            if ds == "DW" and name.lower() in ("lulc", "band1"):
                vals, counts = np.unique(v, return_counts=True)
                row["all_integer"] = bool(np.all(v == np.round(v)))
                row["n_outside_0_8"] = int(((v < 0) | (v > 8)).sum())
                row["class_values"] = " ".join(str(int(x)) for x in vals[:20])
                row["class_pct"] = " ".join(f"{int(x)}:{100*c/v.size:.2f}" for x, c in zip(vals[:20], counts[:20]))
            band_rows.append(row)
            del a, v

    print(f"done  {ds} {res:>2} m  month {month:02d}  {fname}")

files_df = pd.DataFrame(file_rows).sort_values(["dataset", "res_m", "month"])
bands_df = pd.DataFrame(band_rows).sort_values(["dataset", "res_m", "month", "band"])
files_df.to_csv(f"{OUT}/file_report.csv", index=False)
bands_df.to_csv(f"{OUT}/band_report.csv", index=False)
print(f"\nSaved:\n  {OUT}/file_report.csv  ({len(files_df)} files)\n  {OUT}/band_report.csv  ({len(bands_df)} bands)")

done  DW 10 m  month 01  DW_2025_01_10m_9class.tif
done  DW 40 m  month 01  DW_2025_01_40m_9class.tif
done  DW 10 m  month 02  DW_2025_02_10m_9class.tif
done  DW 40 m  month 02  DW_2025_02_40m_9class.tif
done  DW 10 m  month 03  DW_2025_03_10m_9class.tif
done  DW 40 m  month 03  DW_2025_03_40m_9class.tif
done  DW 10 m  month 04  DW_2025_04_10m_9class.tif
done  DW 40 m  month 04  DW_2025_04_40m_9class.tif
done  DW 10 m  month 05  DW_2025_05_10m_9class.tif
done  DW 40 m  month 05  DW_2025_05_40m_9class.tif
done  DW 10 m  month 06  DW_2025_06_10m_9class.tif
done  DW 40 m  month 06  DW_2025_06_40m_9class.tif
done  DW 10 m  month 07  DW_2025_07_10m_9class.tif
done  DW 40 m  month 07  DW_2025_07_40m_9class.tif
done  DW 10 m  month 08  DW_2025_08_10m_9class.tif
done  DW 40 m  month 08  DW_2025_08_40m_9class.tif
done  DW 10 m  month 09  DW_2025_09_10m_9class.tif
done  DW 40 m  month 09  DW_2025_09_40m_9class.tif
done  DW 10 m  month 10  DW_2025_10_10m_9class.tif
done  DW 40 m  month 10  DW_202

In [2]:
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)

print("=== 1. BANDS, DTYPES, CRS, NODATA per dataset/resolution ===")
print(files_df.groupby(["dataset", "res_m"]).agg(
    files=("file", "count"), bands=("bands", "unique"), descriptions=("descriptions", "unique"),
    dtypes=("dtypes", "unique"), crs=("crs", "unique"), nodata=("nodata", "unique")).to_string())

print("\n=== 2. GRID per dataset/resolution (each column should have exactly ONE value) ===")
grid_cols = ["width", "height", "pixel_x", "pixel_y", "rot_b", "rot_d", "origin_x", "origin_y"]
print(files_df.groupby(["dataset", "res_m"])[grid_cols].agg(lambda s: sorted(set(s))).to_string())

print("\n=== 3. S1 vs DW grid at the same resolution ===")
for res in (10, 40):
    s1 = files_df[(files_df.dataset == "S1") & (files_df.res_m == res)][grid_cols].drop_duplicates()
    dw = files_df[(files_df.dataset == "DW") & (files_df.res_m == res)][grid_cols].drop_duplicates()
    same = len(s1) == 1 and len(dw) == 1 and (s1.values == dw.values).all()
    print(f"{res} m: identical grid = {same}")

print("\n=== 4. 40 m vs 10 m nesting (offsets should be whole numbers of 10 m pixels) ===")
for ds in ("S1", "DW"):
    g10 = files_df[(files_df.dataset == ds) & (files_df.res_m == 10)].iloc[0]
    g40 = files_df[(files_df.dataset == ds) & (files_df.res_m == 40)].iloc[0]
    print(f"{ds}: origin offset x = {(g40.origin_x - g10.origin_x)/10:.3f} px, "
          f"y = {(g10.origin_y - g40.origin_y)/10:.3f} px | "
          f"10 m size {g10.width}x{g10.height}, 40 m size {g40.width}x{g40.height}, "
          f"ratio {g10.width/g40.width:.4f}")

print("\n=== 5. Sentinel-1 value ranges per band (dB / counts), all months ===")
s1b = bands_df[bands_df.dataset == "S1"]
print(s1b.groupby(["res_m", "name"])[["pct_invalid", "min", "p1", "median", "p99", "max"]]
      .agg(["min", "max"]).round(2).to_string())

print("\n=== 6. Dynamic World non-class bands (confidence, purity, n_obs) ===")
dwb = bands_df[(bands_df.dataset == "DW") & ~bands_df.name.str.lower().isin(["lulc", "band1"])]
print(dwb.groupby(["res_m", "name"])[["pct_invalid", "min", "median", "max"]]
      .agg(["min", "max"]).round(3).to_string())

print("\n=== 7. Dynamic World class labels per month ===")
lab = bands_df[(bands_df.dataset == "DW") & bands_df.name.str.lower().isin(["lulc", "band1"])]
print(lab[["res_m", "month", "pct_invalid", "all_integer", "n_outside_0_8", "class_values"]].to_string(index=False))

=== 1. BANDS, DTYPES, CRS, NODATA per dataset/resolution ===
               files bands                    descriptions     dtypes           crs  nodata
dataset res_m                                                                              
DW      10        12   [3]         [lulc|confidence|n_obs]  [float32]  [EPSG:32632]  [None]
        40        12   [4]  [lulc|confidence|n_obs|purity]  [float32]  [EPSG:32632]  [None]
S1      10        12   [3]              [VV|VH|mean_n_obs]  [float32]  [EPSG:32632]  [None]
        40        12   [3]              [VV|VH|mean_n_obs]  [float64]  [EPSG:32632]  [None]

=== 2. GRID per dataset/resolution (each column should have exactly ONE value) ===
                width  height pixel_x  pixel_y  rot_b  rot_d    origin_x     origin_y
dataset res_m                                                                        
DW      10     [4001]  [4001]  [10.0]  [-10.0]  [0.0]  [0.0]  [456410.0]  [5491760.0]
        40     [1001]  [1001]  [40.0]  [-40.0

In [3]:
import os, json, math
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window

PROJ = "/home/jovyan/vault/heidelberg_lulc_2025"
ALIGN = f"{PROJ}/outputs/alignment"
os.makedirs(ALIGN, exist_ok=True)
files_df = pd.read_csv(f"{PROJ}/outputs/inspection/file_report.csv")

# 1. One grid per dataset/resolution (all 12 months must be identical)
grids = {}
for (ds, res), g in files_df.groupby(["dataset", "res_m"]):
    cols = ["width", "height", "pixel_x", "pixel_y", "origin_x", "origin_y"]
    assert len(g[cols].drop_duplicates()) == 1, f"{ds} {res} m: months have different grids!"
    r = g.iloc[0]
    assert r.pixel_x == res and r.pixel_y == -res, f"{ds} {res} m: unexpected pixel size"
    on_lattice = (r.origin_x % res == 0) and (r.origin_y % res == 0)
    grids[f"{ds}_{res}"] = dict(res=int(res), left=float(r.origin_x), top=float(r.origin_y),
                                right=float(r.origin_x + r.width * res),
                                bottom=float(r.origin_y - r.height * res))
    gg = grids[f"{ds}_{res}"]
    print(f"{ds} {res:>2} m: x {gg['left']:.0f}-{gg['right']:.0f}, y {gg['bottom']:.0f}-{gg['top']:.0f}, "
          f"origin on {res} m lattice: {on_lattice}")

# 2. Intersection of all four, snapped INWARD to the 40 m lattice
left   = math.ceil (max(g["left"]   for g in grids.values()) / 40) * 40
right  = math.floor(min(g["right"]  for g in grids.values()) / 40) * 40
top    = math.floor(min(g["top"]    for g in grids.values()) / 40) * 40
bottom = math.ceil (max(g["bottom"] for g in grids.values()) / 40) * 40

common = {"crs": "EPSG:32632", "left": left, "right": right, "top": top, "bottom": bottom,
          "10m": {"width": int((right - left) / 10), "height": int((top - bottom) / 10),
                  "transform": [10, 0, left, 0, -10, top]},
          "40m": {"width": int((right - left) / 40), "height": int((top - bottom) / 40),
                  "transform": [40, 0, left, 0, -40, top]},
          "windows": {}}

# 3. Crop window (in pixels) for each dataset; must be whole numbers
for key, g in grids.items():
    res = g["res"]
    col_off = (left - g["left"]) / res
    row_off = (g["top"] - top) / res
    assert col_off.is_integer() and row_off.is_integer(), f"{key}: window not whole pixels!"
    common["windows"][key] = dict(col_off=int(col_off), row_off=int(row_off),
                                  width=common[f"{res}m"]["width"], height=common[f"{res}m"]["height"])

with open(f"{ALIGN}/common_grid.json", "w") as f:
    json.dump(common, f, indent=2)

print(f"\nCommon grid: x {left}-{right}, y {bottom}-{top}  "
      f"({(right-left)/1000:.2f} km x {(top-bottom)/1000:.2f} km = {(right-left)*(top-bottom)/1e6:.1f} km²)")
print(f"10 m: {common['10m']['width']} x {common['10m']['height']} px | "
      f"40 m: {common['40m']['width']} x {common['40m']['height']} px")
for k, w in common["windows"].items():
    print(f"  window {k:6s}: col_off={w['col_off']}, row_off={w['row_off']}, size {w['width']}x{w['height']}")
print("\nSaved:", f"{ALIGN}/common_grid.json")

DW 10 m: x 456410-496420, y 5451750-5491760, origin on 10 m lattice: True
DW 40 m: x 456400-496440, y 5451720-5491760, origin on 40 m lattice: True
S1 10 m: x 456500-496400, y 5451750-5491770, origin on 10 m lattice: True
S1 40 m: x 456480-496400, y 5451720-5491800, origin on 40 m lattice: True

Common grid: x 456520-496400, y 5451760-5491760  (39.88 km x 40.00 km = 1595.2 km²)
10 m: 3988 x 4000 px | 40 m: 997 x 1000 px
  window DW_10 : col_off=11, row_off=0, size 3988x4000
  window DW_40 : col_off=3, row_off=0, size 997x1000
  window S1_10 : col_off=2, row_off=1, size 3988x4000
  window S1_40 : col_off=1, row_off=1, size 997x1000

Saved: /home/jovyan/vault/heidelberg_lulc_2025/outputs/alignment/common_grid.json


In [4]:
def read_win(path, band, key):
    w = common["windows"][key]
    with rasterio.open(path) as src:
        return src.read(band, window=Window(w["col_off"], w["row_off"], w["width"], w["height"])).astype(np.float32)

# ---------- Dynamic World: 10 m majority vs exported 40 m ----------
dw10 = f"{PROJ}/dynamic_world/raw/DW_2025_06_10m_9class.tif"
dw40 = f"{PROJ}/dynamic_world/raw/DW_2025_06_40m_9class.tif"
l10 = read_win(dw10, 1, "DW_10")
l40 = read_win(dw40, 1, "DW_40")
p40 = read_win(dw40, 4, "DW_40")            # purity band

H, W = l40.shape
blocks = l10.reshape(H, 4, W, 4)
counts = np.stack([(blocks == c).sum(axis=(1, 3)) for c in range(9)])     # (9, H, W)
nvalid = counts.sum(0)
majority = counts.argmax(0)
ties = (counts == counts.max(0)).sum(0) > 1
ok = (nvalid > 0) & np.isfinite(l40)

agree_all = (majority[ok] == l40[ok]).mean()
agree_no_ties = (majority[ok & ~ties] == l40[ok & ~ties]).mean()
purity_calc = counts.max(0) / np.maximum(nvalid, 1)
pur_diff = np.abs(purity_calc[ok] - p40[ok])

print("=== Dynamic World June: rebuilt 40 m (majority of 4x4) vs exported 40 m ===")
print(f"agreement, all pixels        : {100*agree_all:.2f} %")
print(f"agreement, excluding ties    : {100*agree_no_ties:.2f} %   (ties = {100*ties[ok].mean():.2f} % of pixels)")
print(f"purity difference median/p99 : {np.median(pur_diff):.4f} / {np.percentile(pur_diff, 99):.4f}")

# ---------- Sentinel-1: how was the 40 m made? ----------
s10 = f"{PROJ}/sentinel1/raw/S1_2025_Jun_Orbit15_10m.tif"
s40 = f"{PROJ}/sentinel1/raw/S1_2025_Jun_Orbit15_40m.tif"
print("\n=== Sentinel-1 June: exported 40 m vs values computed from 10 m ===")
for band, name in [(1, "VV"), (2, "VH")]:
    v10 = read_win(s10, band, "S1_10")
    v40 = read_win(s40, band, "S1_40")
    lin_mean_db = 10 * np.log10((10 ** (v10 / 10)).reshape(H, 4, W, 4).mean(axis=(1, 3)))
    nearest_db = v10[2::4, 2::4]               # pixel nearest to each 40 m block centre
    m = np.isfinite(v40) & np.isfinite(lin_mean_db)
    d_mean = np.abs(v40[m] - lin_mean_db[m])
    d_near = np.abs(v40[m] - nearest_db[m])
    print(f"{name}: |40m - linear 4x4 mean| median {np.median(d_mean):.3f} dB, p95 {np.percentile(d_mean, 95):.3f} dB | "
          f"|40m - single 10m pixel| median {np.median(d_near):.3f} dB")

=== Dynamic World June: rebuilt 40 m (majority of 4x4) vs exported 40 m ===
agreement, all pixels        : 100.00 %
agreement, excluding ties    : 100.00 %   (ties = 0.95 % of pixels)
purity difference median/p99 : 0.0000 / 0.0000

=== Sentinel-1 June: exported 40 m vs values computed from 10 m ===
VV: |40m - linear 4x4 mean| median 0.131 dB, p95 0.463 dB | |40m - single 10m pixel| median 0.604 dB
VH: |40m - linear 4x4 mean| median 0.134 dB, p95 0.445 dB | |40m - single 10m pixel| median 0.601 dB


In [5]:
from rasterio.transform import Affine

MONTH_ABBR = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
OUTA = f"{PROJ}/outputs/aligned"
report = []

for res in (10, 40):
    os.makedirs(f"{OUTA}/{res}m", exist_ok=True)
    g = common[f"{res}m"]
    transform = Affine(res, 0, common["left"], 0, -res, common["top"])
    base = dict(driver="GTiff", width=g["width"], height=g["height"], crs=common["crs"],
                transform=transform, tiled=True, blockxsize=256, blockysize=256, compress="deflate")
    wS = common["windows"][f"S1_{res}"]
    wD = common["windows"][f"DW_{res}"]

    for m in range(1, 13):
        s1_src = f"{PROJ}/sentinel1/raw/S1_2025_{MONTH_ABBR[m-1]}_Orbit15_{res}m.tif"
        dw_src = f"{PROJ}/dynamic_world/raw/DW_2025_{m:02d}_{res}m_9class.tif"

        with rasterio.open(s1_src) as src:
            s1 = src.read(window=Window(**wS)).astype(np.float32)
            s1_names = list(src.descriptions)
        with rasterio.open(dw_src) as src:
            dw = src.read(window=Window(**wD)).astype(np.float32)
            dw_names = list(src.descriptions)

        # labels: float -> uint8, NaN -> 255, and prove no value changed
        lulc_f = dw[0]
        finite = np.isfinite(lulc_f)
        lab = np.where(finite, lulc_f, 255).astype(np.uint8)
        assert np.array_equal(lab[finite].astype(np.float32), lulc_f[finite]), "label values changed!"
        aux, aux_names = dw[1:], dw_names[1:]

        s1_out  = f"{OUTA}/{res}m/S1_2025_{m:02d}_{res}m.tif"
        lab_out = f"{OUTA}/{res}m/DW_2025_{m:02d}_{res}m_lulc.tif"
        aux_out = f"{OUTA}/{res}m/DW_2025_{m:02d}_{res}m_aux.tif"

        with rasterio.open(s1_out, "w", **base, count=s1.shape[0], dtype="float32",
                           nodata=np.nan, predictor=3) as dst:
            dst.write(s1)
            for i, n in enumerate(s1_names, start=1):
                dst.set_band_description(i, n)
        with rasterio.open(lab_out, "w", **base, count=1, dtype="uint8", nodata=255) as dst:
            dst.write(lab, 1)
            dst.set_band_description(1, "lulc")
        with rasterio.open(aux_out, "w", **base, count=aux.shape[0], dtype="float32",
                           nodata=np.nan, predictor=3) as dst:
            dst.write(aux)
            for i, n in enumerate(aux_names, start=1):
                dst.set_band_description(i, n)

        s1_valid = np.isfinite(s1).all(axis=0)
        dw_valid = lab != 255
        report.append({"res_m": res, "month": m,
                       "s1_valid_pct": round(100 * s1_valid.mean(), 3),
                       "dw_valid_pct": round(100 * dw_valid.mean(), 3),
                       "both_valid_pct": round(100 * (s1_valid & dw_valid).mean(), 3),
                       "dw_only_pct": round(100 * (dw_valid & ~s1_valid).mean(), 3),
                       "s1_only_pct": round(100 * (s1_valid & ~dw_valid).mean(), 3)})
        print(f"written {res:>2} m month {m:02d} | S1 valid {report[-1]['s1_valid_pct']:7.3f} % | "
              f"DW valid {report[-1]['dw_valid_pct']:7.3f} % | both {report[-1]['both_valid_pct']:7.3f} %")
        del s1, dw, lab, aux

pd.DataFrame(report).to_csv(f"{ALIGN}/alignment_report.csv", index=False)
print("\nSaved:", f"{ALIGN}/alignment_report.csv")

written 10 m month 01 | S1 valid 100.000 % | DW valid  42.632 % | both  42.632 %
written 10 m month 02 | S1 valid 100.000 % | DW valid 100.000 % | both 100.000 %
written 10 m month 03 | S1 valid 100.000 % | DW valid  99.927 % | both  99.927 %
written 10 m month 04 | S1 valid 100.000 % | DW valid 100.000 % | both 100.000 %
written 10 m month 05 | S1 valid 100.000 % | DW valid 100.000 % | both 100.000 %
written 10 m month 06 | S1 valid 100.000 % | DW valid 100.000 % | both 100.000 %
written 10 m month 07 | S1 valid 100.000 % | DW valid  99.374 % | both  99.374 %
written 10 m month 08 | S1 valid 100.000 % | DW valid 100.000 % | both 100.000 %
written 10 m month 09 | S1 valid 100.000 % | DW valid  99.606 % | both  99.606 %
written 10 m month 10 | S1 valid 100.000 % | DW valid  86.481 % | both  86.481 %
written 10 m month 11 | S1 valid 100.000 % | DW valid  35.644 % | both  35.644 %
written 10 m month 12 | S1 valid 100.000 % | DW valid 100.000 % | both 100.000 %
written 40 m month 01 | S1 v

In [6]:
import glob

print("=== 1. Every aligned file on the common grid? ===")
problems = 0
for p in sorted(glob.glob(f"{OUTA}/*/*.tif")):
    res = 10 if "/10m/" in p else 40
    g = common[f"{res}m"]
    with rasterio.open(p) as src:
        same = (src.crs.to_string() == common["crs"] and src.width == g["width"] and
                src.height == g["height"] and
                tuple(round(x, 6) for x in src.transform[:6]) == tuple(float(x) for x in g["transform"]))
    if not same:
        problems += 1
        print("  MISMATCH:", os.path.relpath(p, PROJ))
n_files = len(glob.glob(f"{OUTA}/*/*.tif"))
print(f"  {n_files} files checked, {problems} problems (expected 72 files, 0 problems)")

print("\n=== 2. 40 m labels = majority of 10 m labels, every month (aligned files) ===")
for m in range(1, 13):
    with rasterio.open(f"{OUTA}/10m/DW_2025_{m:02d}_10m_lulc.tif") as s:
        a10 = s.read(1)
    with rasterio.open(f"{OUTA}/40m/DW_2025_{m:02d}_40m_lulc.tif") as s:
        a40 = s.read(1)
    H, W = a40.shape
    b = a10.reshape(H, 4, W, 4)
    cnt = np.stack([(b == c).sum(axis=(1, 3)) for c in range(9)])
    ok = (cnt.sum(0) > 0) & (a40 != 255)
    ties = (cnt == cnt.max(0)).sum(0) > 1
    agree = (cnt.argmax(0)[ok & ~ties] == a40[ok & ~ties]).mean()
    print(f"  month {m:02d}: agreement (excl. ties) {100*agree:6.2f} %")

print("\n=== 3. Valid-data summary ===")
print(pd.read_csv(f"{ALIGN}/alignment_report.csv").to_string(index=False))

=== 1. Every aligned file on the common grid? ===
  72 files checked, 0 problems (expected 72 files, 0 problems)

=== 2. 40 m labels = majority of 10 m labels, every month (aligned files) ===
  month 01: agreement (excl. ties) 100.00 %
  month 02: agreement (excl. ties) 100.00 %
  month 03: agreement (excl. ties) 100.00 %
  month 04: agreement (excl. ties) 100.00 %
  month 05: agreement (excl. ties) 100.00 %
  month 06: agreement (excl. ties) 100.00 %
  month 07: agreement (excl. ties) 100.00 %
  month 08: agreement (excl. ties) 100.00 %
  month 09: agreement (excl. ties) 100.00 %
  month 10: agreement (excl. ties) 100.00 %
  month 11: agreement (excl. ties) 100.00 %
  month 12: agreement (excl. ties) 100.00 %

=== 3. Valid-data summary ===
 res_m  month  s1_valid_pct  dw_valid_pct  both_valid_pct  dw_only_pct  s1_only_pct
    10      1         100.0        42.632          42.632          0.0       57.368
    10      2         100.0       100.000         100.000          0.0        0.0